# ⚙️ NeuroScope AI — Notebook 00A: Environment Setup

**Local VS Code setup — run once when starting the project.**

Steps:
1. Verify Python environment + GPU
2. Create folder structure
3. Write master_config.yaml
4. Write neuroscope_loader.py
5. Final check

**Prerequisites (do these manually once before running this notebook):**
```bash
# 1. Create venv
python -m venv C:\Users\tejan\NeuroScope_AI\venv

# 2. Activate
C:\Users\tejan\NeuroScope_AI\venv\Scripts\activate

# 3. Install PyTorch with CUDA
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# 4. Install everything else
pip install monai[all]==1.4.0 nibabel SimpleITK pydicom
pip install "numpy<2" --force-reinstall
pip install albumentations opencv-python
pip install efficientnet-pytorch segmentation-models-pytorch pretrainedmodels
pip install grad-cam onnx onnxruntime-gpu
pip install lifelines pycox pyradiomics
pip install pyyaml xlrd openpyxl kaggle
pip install pandas matplotlib seaborn scikit-learn
pip install fastapi uvicorn anthropic
```

---

## Step 1 · Verify Environment & GPU

In [3]:
import sys, os
import importlib

# ── Project root ─────────────────────────────────────────────────────────
BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'
print(f'Project root : {BASE}')
print(f'Python       : {sys.version}')
print(f'Venv active  : {"NeuroScope" in sys.prefix or "venv" in sys.prefix}')
print()

# ── Package check ─────────────────────────────────────────────────────────
packages = {
    'torch':            'PyTorch',
    'monai':            'MONAI',
    'albumentations':   'Albumentations',
    'cv2':              'OpenCV',
    'nibabel':          'NiBabel',
    'SimpleITK':        'SimpleITK',
    'pydicom':          'pydicom',
    'onnx':             'ONNX',
    'onnxruntime':      'ONNX Runtime',
    'pytorch_grad_cam': 'Grad-CAM',
    'yaml':             'PyYAML',
    'sklearn':          'scikit-learn',
    'lifelines':        'Lifelines',
}

print('Package verification:')
all_ok = True
for mod, name in packages.items():
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, '__version__', 'ok')
        print(f'  ✅ {name:<22} {ver}')
    except Exception:
        print(f'  ❌ {name:<22} NOT FOUND — pip install it')
        all_ok = False

# ── GPU check ─────────────────────────────────────────────────────────────
import torch
print()
if torch.cuda.is_available():
    print(f'  🎮 GPU  : {torch.cuda.get_device_name(0)}')
    try:
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f'  🎮 VRAM : {vram:.1f} GB')
    except AttributeError:
        pass
else:
    print('  ⚠️  CUDA not available — check CUDA Toolkit installation')

# ── Numpy check ───────────────────────────────────────────────────────────
import numpy as np
nv = int(np.__version__.split('.')[0])
if nv >= 2:
    print(f'\n  ⚠️  numpy {np.__version__} — run: pip install "numpy<2" --force-reinstall')
else:
    print(f'  ✅ numpy {np.__version__}')

print()
print('✅ All packages OK' if all_ok else '❌ Some packages missing — install them first')

Project root : C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI
Python       : 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Venv active  : False

Package verification:
  ✅ PyTorch                2.10.0+cu126
  ❌ MONAI                  NOT FOUND — pip install it
  ✅ Albumentations         2.0.8
  ✅ OpenCV                 4.13.0
  ❌ NiBabel                NOT FOUND — pip install it
  ❌ SimpleITK              NOT FOUND — pip install it
  ✅ pydicom                3.0.1
  ❌ ONNX                   NOT FOUND — pip install it
  ✅ ONNX Runtime           1.25.1
  ✅ Grad-CAM               ok
  ✅ PyYAML                 6.0.2
  ✅ scikit-learn           1.7.2
  ❌ Lifelines              NOT FOUND — pip install it

  🎮 GPU  : NVIDIA GeForce RTX 4060 Laptop GPU
  🎮 VRAM : 8.0 GB

  ⚠️  numpy 2.3.5 — run: pip install "numpy<2" --force-reinstall

❌ Some packages missing — install them first


---
## Step 2 · Create Folder Structure

In [6]:
FOLDERS = [
    # Core
    rf'{BASE}\configs',
    rf'{BASE}\logs',
    rf'{BASE}\outputs',
    rf'{BASE}\reports',
    rf'{BASE}\notebooks',
    rf'{BASE}\feedback',
    # Datasets
    rf'{BASE}\datasets\brain\brats2024',
    rf'{BASE}\datasets\brain\kaggle_brain_tumor',
    rf'{BASE}\datasets\brain\utsw_glioma',
    rf'{BASE}\datasets\lung\lidc_idri',
    rf'{BASE}\datasets\lung\luna16',
    rf'{BASE}\datasets\lung\nsclc_radiomics',
    rf'{BASE}\datasets\breast\cbis_ddsm',
    rf'{BASE}\datasets\breast\inbreast',
    rf'{BASE}\datasets\breast\cmmd',
    rf'{BASE}\datasets\breast\kau_bcmd',
    rf'{BASE}\datasets\liver\lits',
    rf'{BASE}\datasets\skin\ham10000',
    rf'{BASE}\datasets\skin\isic2020',
    rf'{BASE}\datasets\skin\ph2',
    rf'{BASE}\datasets\spine\rsna_lumbar',
    # Checkpoints
    rf'{BASE}\checkpoints\brain_seg',
    rf'{BASE}\checkpoints\brain_cls',
    rf'{BASE}\checkpoints\lung_det',
    rf'{BASE}\checkpoints\liver_seg',
    rf'{BASE}\checkpoints\breast_det',
    rf'{BASE}\checkpoints\skin_cls',
    rf'{BASE}\checkpoints\spine_cls',
    rf'{BASE}\checkpoints\ensemble',
    # Models
    rf'{BASE}\models\production\brain_seg',
    rf'{BASE}\models\production\brain_cls',
    rf'{BASE}\models\production\lung_det',
    rf'{BASE}\models\production\liver_seg',
    rf'{BASE}\models\production\breast_det',
    rf'{BASE}\models\production\skin_cls',
    rf'{BASE}\models\production\spine_cls',
    rf'{BASE}\models\production\verification_gate',
    rf'{BASE}\models\weights',
    # Source code
    rf'{BASE}\src\agents',
    rf'{BASE}\src\api',
    rf'{BASE}\src\models',
    rf'{BASE}\src\clinical',
    rf'{BASE}\src\explainability',
    rf'{BASE}\src\ethics',
]

created = 0
for folder in FOLDERS:
    if not os.path.exists(folder):
        os.makedirs(folder, exist_ok=True)
        created += 1

print(f'✅ Folder structure ready ({created} new folders created)')

✅ Folder structure ready (27 new folders created)


---
## Step 3 · Write Master Config

In [11]:
import yaml

config = {
    'project': {
        'name': 'NeuroScope AI',
        'version': '2.0',
        'base_path': BASE,
    },
    'data': {
        'root': rf'{BASE}\datasets',
    },
    'paths': {
        'datasets': {
            'brain_brats':     rf'{BASE}\datasets\brain\brats2024',
            'brain_kaggle':    rf'{BASE}\datasets\brain\kaggle_brain_tumor',
            'brain_utsw':      rf'{BASE}\datasets\brain\utsw_glioma',
            'lung_lidc':       rf'{BASE}\datasets\lung\lidc_idri',
            'lung_luna':       rf'{BASE}\datasets\lung\luna16',
            'lung_nsclc':      rf'{BASE}\datasets\lung\nsclc_radiomics',
            'breast_cbis':     rf'{BASE}\datasets\breast\cbis_ddsm',
            'breast_inbreast': rf'{BASE}\datasets\breast\inbreast',
            'breast_cmmd':     rf'{BASE}\datasets\breast\cmmd',
            'breast_kau':      rf'{BASE}\datasets\breast\kau_bcmd',
            'liver_lits':      rf'{BASE}\datasets\liver\lits',
            'skin_ham':        rf'{BASE}\datasets\skin\ham10000',
            'skin_isic':       rf'{BASE}\datasets\skin\isic2020',
            'skin_ph2':        rf'{BASE}\datasets\skin\ph2',
            'spine_rsna':      rf'{BASE}\datasets\spine\rsna_lumbar',
        },
        'checkpoints': rf'{BASE}\checkpoints',
        'models':      rf'{BASE}\models',
        'outputs':     rf'{BASE}\outputs',
        'logs':        rf'{BASE}\logs',
        'src':         rf'{BASE}\src',
    },
    'training': {
        'device': 'cuda',
        'precision': 'fp16',
        'seed': 42,
        'default_workers': 4,
    },
    'segmentation': {
        'brain': {
            'model': 'SegResNet',
            'num_classes': 4,
            'class_names': ['background', 'NCR', 'edema', 'enhancing_tumor'],
            'roi_size': [128, 128, 128],
            'in_channels': 4,
            'label_remap': {4: 3},
        },
        'liver': {
            'model': 'SegResNet',
            'num_classes': 3,
            'class_names': ['background', 'liver', 'tumor'],
            'hu_window': [-200, 300],
        },
    },
    'classification': {
        'brain_type':  {'classes': ['glioma', 'meningioma', 'notumor', 'pituitary'], 'num_classes': 4},
        'brain_grade': {'classes': ['grade_I', 'grade_II', 'grade_III', 'grade_IV'], 'num_classes': 4},
        'skin':        {'classes': ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc'], 'num_classes': 7},
        'breast':      {'classes': ['benign', 'malignant'], 'num_classes': 2},
        'spine': {
            'conditions': [
                'spinal_canal_stenosis',
                'left_neural_foraminal_narrowing',
                'right_neural_foraminal_narrowing',
                'left_subarticular_stenosis',
                'right_subarticular_stenosis',
            ],
            'levels':   ['L1_L2', 'L2_L3', 'L3_L4', 'L4_L5', 'L5_S1'],
            'severity': ['normal_mild', 'moderate', 'severe'],
        },
    },
    'molecular': {
        'markers': ['idh_mutation', 'mgmt_methylation', '1p19q_codeletion'],
    },
    'agents': {
        'verification_gate_thresholds': {
            'brain':         0.97,
            'lung':          0.95,
            'breast':        0.95,
            'liver':         0.93,
            'skin_melanoma': 0.99,
            'skin_other':    0.95,
            'spine':         0.95,
        },
        'emergency_response_seconds': 30,
    },
}

config_path = os.path.join(BASE, 'configs', 'master_config.yaml')
with open(config_path, 'w', encoding='utf-8') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
print(f'✅ Master config created at: {config_path}')

✅ Master config created at: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI\configs\master_config.yaml


---
## Step 4 · Write neuroscope_loader.py

In [13]:
import os

BASE = r'C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI'

loader_code = (
    '"""NeuroScope AI — Loader Utility (auto-generated by Notebook 00A)\n'
    '\n'
    'Usage at the top of every notebook:\n'
    '    import sys\n'
    '    sys.path.insert(0, r\'C:\\\\Users\\\\tejan\\\\OneDrive\\\\Desktop\\\\drive\\\\NeuroScope_AI\')\n'
    '    from neuroscope_loader import cfg, DEVICE, BASE\n'
    '"""\n'
    'import os, yaml, torch\n'
    '\n'
    'BASE = r\'C:\\Users\\tejan\\OneDrive\\Desktop\\drive\\NeuroScope_AI\'\n'
    '\n'
    'with open(os.path.join(BASE, "configs", "master_config.yaml"), encoding="utf-8") as f:\n'
    '    cfg = yaml.safe_load(f)\n'
    '\n'
    'DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
    '\n'
    '\n'
    'def save_checkpoint(model, optimizer, epoch, loss, path, scheduler=None, extra=None):\n'
    '    """Save training checkpoint."""\n'
    '    os.makedirs(os.path.dirname(path), exist_ok=True)\n'
    '    state = {\n'
    '        "epoch": epoch, "loss": loss,\n'
    '        "model_state_dict": model.state_dict(),\n'
    '        "optimizer_state_dict": optimizer.state_dict(),\n'
    '    }\n'
    '    if scheduler:\n'
    '        state["scheduler_state_dict"] = scheduler.state_dict()\n'
    '    if extra:\n'
    '        state.update(extra)\n'
    '    torch.save(state, path)\n'
    '\n'
    '\n'
    'def load_checkpoint(path, model, optimizer, scheduler=None):\n'
    '    """Resume from checkpoint. Returns next epoch number."""\n'
    '    if not os.path.exists(path):\n'
    '        print("  No checkpoint found - starting from scratch")\n'
    '        return 0\n'
    '    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)\n'
    '    model.load_state_dict(ckpt["model_state_dict"])\n'
    '    optimizer.load_state_dict(ckpt["optimizer_state_dict"])\n'
    '    if scheduler and "scheduler_state_dict" in ckpt:\n'
    '        scheduler.load_state_dict(ckpt["scheduler_state_dict"])\n'
    '    print(f\'  Resumed epoch {ckpt["epoch"]}, loss {ckpt["loss"]:.4f}\')\n'
    '    return ckpt["epoch"] + 1\n'
    '\n'
    '\n'
    'gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"\n'
    'print(f"NeuroScope AI | Device: {gpu} | Root: {BASE}")\n'
)

loader_path = os.path.join(BASE, 'neuroscope_loader.py')
with open(loader_path, 'w', encoding='utf-8') as f:
    f.write(loader_code)

# Verify it imports cleanly
import importlib.util
spec = importlib.util.spec_from_file_location('neuroscope_loader', loader_path)
mod  = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
print('neuroscope_loader.py saved and verified')

NeuroScope AI | Device: NVIDIA GeForce RTX 4060 Laptop GPU | Root: C:\Users\tejan\OneDrive\Desktop\drive\NeuroScope_AI
neuroscope_loader.py saved and verified


---
## Step 5 · Final Check

In [14]:
import torch, os

print('=' * 55)
print('  NEUROSCOPE AI — SETUP COMPLETE')
print('=' * 55)

checks = [
    ('master_config.yaml',   os.path.join(BASE, 'configs', 'master_config.yaml')),
    ('neuroscope_loader.py', os.path.join(BASE, 'neuroscope_loader.py')),
    ('datasets/',            os.path.join(BASE, 'datasets')),
    ('checkpoints/',         os.path.join(BASE, 'checkpoints')),
    ('models/',              os.path.join(BASE, 'models')),
    ('src/',                 os.path.join(BASE, 'src')),
    ('outputs/',             os.path.join(BASE, 'outputs')),
]

for label, path in checks:
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {label}')

print()
if torch.cuda.is_available():
    print(f'  🎮 {torch.cuda.get_device_name(0)}')

print()
print('  Next: Run Notebook 00B to verify datasets')
print('  Then: Notebook 01 — Data Exploration')
print('=' * 55)

  NEUROSCOPE AI — SETUP COMPLETE
  ✅ master_config.yaml
  ✅ neuroscope_loader.py
  ✅ datasets/
  ✅ checkpoints/
  ✅ models/
  ✅ src/
  ✅ outputs/

  🎮 NVIDIA GeForce RTX 4060 Laptop GPU

  Next: Run Notebook 00B to verify datasets
  Then: Notebook 01 — Data Exploration
